In [ ]:
# Apre la finestra di upload di Colab per caricare manualmente i file con i dati
from google.colab import files
uploaded = files.upload()

Saving Comuni - Dimensione Data Indagine 11-07-2026 Stampa 11072026101548.csv to Comuni - Dimensione Data Indagine 11-07-2026 Stampa 11072026101548.csv
Saving Comuni - Caratteristiche del territorio Data Indagine 11-07-2026 Stampa 11072026101340.csv to Comuni - Caratteristiche del territorio Data Indagine 11-07-2026 Stampa 11072026101340.csv
Saving gi_db_comuni-2026-07-10-b893f.zip to gi_db_comuni-2026-07-10-b893f.zip
Saving Altimetria_Comuni-al-31_12_2021.xlsx to Altimetria_Comuni-al-31_12_2021.xlsx


In [ ]:
# Estrae il contenuto dello zip caricato "gi_db_comuni"
import zipfile
with zipfile.ZipFile("gi_db_comuni-2026-07-10-b893f.zip", "r") as z:
    z.extractall("gi_db_comuni")

In [ ]:
# Verifica che lo zip sia estratto correttamente
import os
print(os.listdir())
print(os.listdir("gi_db_comuni"))

['.config', 'gi_db_comuni-2026-07-10-b893f.zip', 'gi_db_comuni', 'Altimetria_Comuni-al-31_12_2021.xlsx', 'Comuni - Caratteristiche del territorio Data Indagine 11-07-2026 Stampa 11072026101340.csv', 'Comuni - Dimensione Data Indagine 11-07-2026 Stampa 11072026101548.csv', 'sample_data']
['xlsx', 'csv', 'README.txt', 'json', 'mysql']


In [ ]:
import re
import pandas as pd

BASE_DIR = "."

# ---------------------------------------------------------------------------
# 1. Caricamento file di partenza: Caratteristiche del territorio
# ---------------------------------------------------------------------------
car = pd.read_csv(
    f"{BASE_DIR}/Comuni - Caratteristiche del territorio Data Indagine 11-07-2026 Stampa 11072026101340.csv",
    sep=";",
    dtype=str,
)

# Elimina le colonne H e I: H = "Comune (dizione straniera)", I = "Sigla automobilistica"
col_H = car.columns[7]
col_I = car.columns[8]
assert col_H == "Comune (dizione straniera)"
assert col_I == "Sigla automobilistica"
car = car.drop(columns=[col_H, col_I])

# Elimina il  "Codice comune (numerico)": ridondante con "Codice Comune (alfanumerico)" -> tengo solo quello a 6 cifre.
car = car.drop(columns=["Codice comune (numerico)"])

# Crea una chiave di join
car["_key"] = car["Codice Comune (alfanumerico)"].str.strip()

# ---------------------------------------------------------------------------
# 2. Dal file "Comuni - Dimensione Data Indagine 11-07-2026" prende la colonna "Superficie (Kmq)" e la aggiungo al dataframe
# ---------------------------------------------------------------------------
# Carica il file "Comuni - Dimensione Data Indagine 11-07-2026"
dim = pd.read_csv(
    f"{BASE_DIR}/Comuni - Dimensione Data Indagine 11-07-2026 Stampa 11072026101548.csv",
    sep=";",
    dtype=str,
)
# Crea una chiave di join
dim["_key"] = dim["Codice Comune (alfanumerico)"].str.strip()

# Tiene dal file "Dimensione" la colonna"Superficie (Kmq)"
dim_sup = dim[["_key", "Superficie (Kmq)"]].drop_duplicates(subset="_key")

#Unisce la colonna "Superficie(kmq)" al dataframe "Car"
car = car.merge(dim_sup, on="_key", how="left")

# ---------------------------------------------------------------------------
# 3. Dal file "gi_cap.csv" -> tutte le colonne (un comune può avere più CAP: aggregati
#    in un'unica stringa separata da virgola)
# ---------------------------------------------------------------------------
# Carica il file "gi_cap.csv"
cap = pd.read_csv(f"{BASE_DIR}/gi_db_comuni/csv/gi_cap.csv", sep=";", dtype=str)

# Crea la chiave di join, togliendo dal "codice_istat" gli spazi
cap["_key"] = cap["codice_istat"].str.strip()

# Raggruppa tutte le righe che hanno la stessa key, isolando "cap". Toglie i duplicati, li ordina e li unisce in un'unica stringa
cap_agg = (
    cap.groupby("_key")["cap"]
    .apply(lambda s: ", ".join(sorted(s.unique())))
    .reset_index()
)

# Unisce la colonna "cap" al dataframe "car"
car = car.merge(cap_agg, on="_key", how="left")

# ---------------------------------------------------------------------------
# 4. gi_comuni.csv -> sigla_provincia, lat, lon
# ---------------------------------------------------------------------------
gc = pd.read_csv(f"{BASE_DIR}/gi_db_comuni/csv/gi_comuni.csv", sep=";", dtype=str)
gc["_key"] = gc["codice_istat"].str.strip()
gc_sel = gc[["_key", "sigla_provincia", "lat", "lon"]].drop_duplicates(subset="_key")

car = car.merge(gc_sel, on="_key", how="left")

# ---------------------------------------------------------------------------
# 5. gi_comuni_validita.csv -> data_inizio_validita, data_fine_validita,
#    stato_validita (solo il record "Attivo" per ogni comune)
# ---------------------------------------------------------------------------
val = pd.read_csv(f"{BASE_DIR}/gi_db_comuni/csv/gi_comuni_validita.csv", sep=";", dtype=str)
val_attivi = val[val["stato_validita"] == "Attivo"].copy()
val_attivi["_key"] = val_attivi["codice_istat"].str.strip()
val_sel = val_attivi[
    ["_key", "data_inizio_validita", "data_fine_validita", "stato_validita"]
].drop_duplicates(subset="_key")

car = car.merge(val_sel, on="_key", how="left")

# ---------------------------------------------------------------------------
# 6. gi_province.csv -> numero_comuni (quanti comuni ci sono nella stessa
#    provincia/UTS del comune in questione)
#    join tramite "Codice Provincia/Uts" <-> "codice_sovracomunale"
# ---------------------------------------------------------------------------
prov = pd.read_csv(f"{BASE_DIR}/gi_db_comuni/csv/gi_province.csv", sep=";", dtype=str)
prov_sel = prov[["codice_sovracomunale", "numero_comuni"]].drop_duplicates(
    subset="codice_sovracomunale"
)

car = car.merge(
    prov_sel,
    left_on="Codice Provincia/Uts",
    right_on="codice_sovracomunale",
    how="left",
)
car = car.drop(columns=["codice_sovracomunale"])

# ---------------------------------------------------------------------------
# Pulizia colonna di servizio
# ---------------------------------------------------------------------------
car = car.drop(columns=["_key"])

# ---------------------------------------------------------------------------
# Rinomina colonne: spazi, "-" e "/" diventano "_", tutto minuscolo,
# tolte le parentesi
# ---------------------------------------------------------------------------
def slugify(col):
    c = col.strip()
    c = c.replace("/", "_")
    c = c.replace("-", "_")
    c = c.replace("(", "").replace(")", "")
    c = re.sub(r"\s+", "_", c)
    c = re.sub(r"_+", "_", c)
    return c.lower()

car.columns = [slugify(c) for c in car.columns]

# Rinomine specifiche richieste:
# - Codice Provincia/Uts (colonna C originale)      -> uts
# - Codice Provincia (Storico) (colonna D originale) -> codice_provincia
# - Codice Comune (alfanumerico) (colonna E originale) -> codice_istat
car = car.rename(columns={
    "codice_provincia_uts": "uts",
    "codice_provincia_storico": "codice_provincia",
    "codice_comune_alfanumerico": "codice_istat",
})

# Verifica finale: tutte le colonne devono essere di tipo stringa (object)
non_string_cols = [c for c in car.columns if car[c].dropna().map(type).ne(str).any()]
assert not non_string_cols, f"Colonne non in formato stringa: {non_string_cols}"

output_path = f"{BASE_DIR}/comuni_unito.csv"
car.to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")

print("Righe totali:", len(car))
print("Colonne finali:", car.columns.tolist())
print("Comuni senza numero_comuni:", car["numero_comuni"].isna().sum())
print("Comuni senza stato_validita:", car["stato_validita"].isna().sum())
print("File salvato in:", output_path)

Righe totali: 7894
Colonne finali: ['codice_ripartizione_geografica', 'codice_regione', 'uts', 'codice_provincia', 'codice_istat', 'comune', 'comune_isolano', 'comune_litoraneo', 'zona_altimetrica', 'altitudine_municipio', 'zone_costiere_2021', 'degurba_2021', 'codice_ecoregioni_divisioni', 'ecoregioni_divisioni', 'codice_ecoregioni_province', 'ecoregioni_province', 'codice_ecoregioni_sezioni', 'ecoregioni_sezioni', 'codice_ecoregioni_sottosezioni', 'ecoregioni_sottosezioni', 'superficie_kmq', 'cap', 'sigla_provincia', 'lat', 'lon', 'data_inizio_validita', 'data_fine_validita', 'stato_validita', 'numero_comuni']
Comuni senza numero_comuni: 0
Comuni senza stato_validita: 1
File salvato in: ./comuni_unito.csv


In [ ]:
pd.read_csv("/content/comuni_unito.csv",sep=";",dtype=str)

,codice_ripartizione_geografica,codice_regione,uts,codice_provincia,codice_istat,comune,comune_isolano,comune_litoraneo,zona_altimetrica,altitudine_municipio,...,ecoregioni_sottosezioni,superficie_kmq,cap,sigla_provincia,lat,lon,data_inizio_validita,data_fine_validita,stato_validita,numero_comuni
0,1,01,201,001,001001,Agliè,0,0,3,333,...,Sottosezione Bacino Occidentale del Po,"13,1463",10011,TO,"45,3634669","7,7686057",05/10/1945,NaN,Attivo,312
1,1,01,201,001,001002,Airasca,0,0,5,261,...,Sottosezione Bacino Occidentale del Po,"15,7393",10060,TO,"44,9170061","7,4845045",26/05/1893,NaN,Attivo,312
2,1,01,201,001,001003,Ala di Stura,0,0,1,1073,...,Sottosezione Alpi Nord-Occidentali,"46,3316",10070,TO,"45,3149237","7,3043670",11/03/1864,NaN,Attivo,312
3,1,01,201,001,001004,Albiano d'Ivrea,0,0,3,233,...,Sottosezione Alpi Nord-Occidentali,"11,7397",10010,TO,"45,4336463","7,9495045",05/10/1945,NaN,Attivo,312
4,1,01,201,001,001006,Almese,0,0,3,375,...,Sottosezione Alpi Nord-Occidentali,"17,8741",10040,TO,"45,1176654","7,3951859",17/03/1861,NaN,Attivo,312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7889,5,20,119,119,119020,Sant'Antioco,1,1,4,10,...,Sottosezione Sarda Sud-Occidentale,"88,0162",09017,CI,"39,0664348","8,4554211",18/03/2003,NaN,Attivo,24
7890,5,20,119,119,119021,Tratalias,0,0,4,38,...,Sottosezione Sarda Sud-Occidentale,"31,0025",09010,CI,"39,1030286","8,5775839",18/03/2003,NaN,Attivo,24
7891,5,20,119,119,119022,Villamassargia,0,0,3,127,...,Sottosezione Sarda Sud-Occidentale,"91,3873",09010,CI,"39,2755752","8,6405778",18/03/2003,NaN,Attivo,24
7892,5,20,119,119,119023,Villaperuccio,0,0,3,66,...,Sottosezione Sarda Sud-Occidentale,"36,4281",09010,CI,"39,1117916","8,6719680",18/03/2003,NaN,Attivo,24
